In [480]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from scipy.optimize import minimize

### Data preprocessing

The training data is loaded and the target variable `Sleep_Duration` is separated from the predictors. Numerical features are median-imputed and standardized without centering, while categorical features are imputed using their most frequent value and one-hot encoded. The resulting preprocessing pipeline is then fitted to transform the data into the feature matrix `X`.

In [509]:
data = pd.read_csv("bern-02-2026/train.csv")

df = data.drop(columns=["Sleep_Duration"])
y = np.array(data["Sleep_Duration"])

cat_cols = df.select_dtypes(include="object").columns
num_cols = df.select_dtypes(include="number").columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False))
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X = preprocessor.fit_transform(df)

C:\Users\krane\AppData\Local\Temp\ipykernel_28296\816043707.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


In [516]:
test_data = pd.read_csv("bern-02-2026/test.csv")
test_X = preprocessor.fit_transform(test_data)

### Helper functions

The following functions are defined for use in the modelling and evaluation steps later in the analysis. They handle data splitting, model prediction, mixed L1/L2 regularized regression, residual sum of squares (RSS), model training, k-fold cross-validation, and adjusted \(R^2\).

In [529]:
def split(x, y, train_ratio, random_seed):
    "Splits the data according to train_ratio % : (1-train_ration) % [train:test]. Assumes y is 1D"
    rng = np.random.default_rng(random_seed)
    idx = rng.permutation(len(y))
    
    x_randomized = x[idx]
    y_randomized = y[idx]

    ntrain = int(round(train_ratio*len(x)))
    xtrain, xtest = [x_randomized[:ntrain],x_randomized[ntrain:]] 
    ytrain, ytest = [y_randomized[:ntrain],y_randomized[ntrain:]]
    return xtrain, ytrain, xtest, ytest

def model (betas, x):
    if x is None:
        return betas[0]
    else:
        return betas[0] + x @ betas[1:]
        
def mixed_regression (betas, x, y, lambdal1 = 0.1111111111111111, lambdal2= 0.555555555):
    "A combination of lasso and ridge regression"
    rss = np.sum( (y - model(betas,x))**2 )
    l2  = np.sum( betas[1:]**2 )
    l1  = np.sum( np.abs(betas[1:]) )
    return rss + lambdal1*l1 + lambdal2*l2

def rss(y, yhat):
    "Residual sum squared"
    return np.sum( (y-yhat)**2 )

def train (xtrain, ytrain, objective):
    "Trains a model using linear regression"
    nbetas = xtrain.shape[1] if not None else 1 # does not include b_0
    betas  = minimize(lambda betas: objective(betas, xtrain, ytrain), np.zeros((nbetas + 1))).x
    trss   = rss(ytrain, model(betas, xtrain))
    return betas, trss

def kfold_validate(x, y, f_hat, K=3, random_seed= None):
    "Divides the data into k folds"
    rng = np.random.default_rng(random_seed)
    idx = rng.permutation(len(y))

    x_randomized = np.array(x[idx])
    y_randomized = np.array(y[idx])

    assert len(x)%K == 0 #Ensure that K will equally divide the data!
    cutoff = len(x)//K
    
    idx  = list(range(len(y)))
    kMSEs = np.zeros((K),dtype=float)
    
    for k in range(K):
        val_idx   = idx[k*cutoff:(k+1)*cutoff]
        train_idx = idx[:k*cutoff] + idx[(k+1)*cutoff:]
        
        xval   = x_randomized[val_idx]
        yval   = y_randomized[val_idx]
        
        kMSEs[k]= np.mean( (yval - f_hat(xval))**2 )

    MSE = np.mean(kMSEs)
    return MSE

def adjustedR2 (x, y, f_hat, d):
    RSS = np.sum( (y-f_hat(x))**2   )
    TSS = np.sum( (y-np.mean(y))**2 )
    n = len(y)
    return 1 - ((RSS/(n-d-1))/(TSS/(n-1)))

def predict_save (betas, predictors, tag, save = False):
    X_selected = test_X[:, predictors]

    output = test_data[["ID"]].copy()
    output["Sleep_Duration"] = betas[0] + X_selected @ betas[1:]

    if save:
        output.to_csv(f"predictions_{tag}.csv", index=False)

    return output

### Choosing the regularization parameters

The function `kfold_validate_lambda` evaluates a pair of regularization parameters \(\lambda_1\) and \(\lambda_2\) using 3-fold cross-validation. For each fold, the model is trained using the mixed L1/L2 objective and its validation MSE is calculated. The average validation MSE across the folds is then used to measure how well that parameter combination performs.

Here, `l1s` and `l2s` define possible values for the L1 (Lasso) and L2 (Ridge) regularization strengths. All combinations of these values are tested, and their corresponding cross-validation MSEs are stored in `MSEs`. The goal is to find the \(\lambda_1,\lambda_2\) combination that gives the lowest validation MSE.

In [530]:
def kfold_validate_lambda(x, y, lambdas, K=3, random_seed= None, plot=False):
    "Divides the data into k folds"
    rng = np.random.default_rng(random_seed)
    idx = rng.permutation(len(y))

    x_randomized = np.array(x[idx])
    y_randomized = np.array(y[idx])

    assert len(x)%K == 0 #Ensure that K will equally divide the data!
    cutoff = len(x)//K
    
    idx  = list(range(len(x)))
    kMSEs = np.zeros((K),dtype=float)
    for k in range(K):
        val_idx   = idx[k*cutoff:(k+1)*cutoff]
        train_idx = idx[:k*cutoff] + idx[(k+1)*cutoff:]
        
        xval   = x_randomized[val_idx]
        yval   = y_randomized[val_idx]
        xtrain = x_randomized[train_idx]
        ytrain = y_randomized[train_idx]
        
        objective = lambda betas: mixed_regression(betas, xtrain, ytrain, lambdal1 = lambdas[0], lambdal2 = lambdas[1])
        betas     = minimize(objective, np.zeros((x.shape[1]+1))).x
        kMSEs[k]= np.mean( (yval - model(betas, xval))**2 )

    MSE = np.mean(kMSEs)
    return MSE

l1s = np.linspace(0,1,10)
l2s = np.linspace(0,1,10)

Lambdas = list(combinations(np.concatenate((l1s,l2s)), 2))
MSEs = []
               
for i,lambdas in enumerate(Lambdas):
    print(f"Progress: {(i+1)/len(Lambdas) * 100 :.2f} %", end="\r", flush = True)
    MSE = kfold_validate_lambda(X, y, lambdas)
    MSEs.append(MSE)
    
idx = np.argmin(np.array(MSEs))
print(MSEs[idx], Lambdas[idx])

0.08628130239115113 (np.float64(0.1111111111111111), np.float64(1.0))


In [531]:
l1, l2 = Lambdas[idx]

In this task, forward selection is used because it provides a computationally efficient way of exploring different model sizes while avoiding the large number of models required by best subset selection. Backward selection was also tested and it was concluded that the model does not perform well when fitted with all of the predictors.

### Forward stepwise feature selection

The function `forward_stepwise` builds regression models incrementally by adding one predictor at a time. The data is first split into training and validation sets. The algorithm starts with an intercept-only model and then, at each step, tests every remaining predictor by temporarily adding it to the current model.

For each candidate predictor, a model is fitted using mixed regularization, and the predictor that gives the lowest training RSS is selected. This process continues until all predictors have been added.

After each step, the resulting model is evaluated on the validation data using cross-validation MSE and adjusted \(R^2\). This allows the models with different numbers of predictors to be compared.

Finally, two models are selected:
- the model with the **lowest cross-validation MSE**
- the model with the **highest adjusted \(R^2\)**

The function returns the coefficients and selected predictors for both models.

In [547]:
def forward_stepwise (X, y):
    #Split training and validation data
    xtrain, ytrain, xval, yval = split(X, y, 0.8, None)
    
    p = X.shape[1] # number of predictors
    index = list(range(p))
    CV  = np.zeros((p+1))
    AR2 = np.zeros((p+1))
    
    #Start with no predictors
    predictors = [[]]    
    betas  = [[np.mean(ytrain)]]
    CV[0]  = kfold_validate(xval, yval, lambda x: betas[0])
    AR2[0] = adjustedR2(xval, yval, lambda x: betas[0], 0)
    
    for k in range(1, p+1):
        #initiate variables
        old_rss  = 1e10
        new_beta = None
        idx      = None
        
        for i in index:
            
            train = xtrain[:,predictors[k-1] + [i]]
            
            objective = lambda betas: mixed_regression(betas, train, ytrain, lambdal1 = l1, lambdal2 = l2)
            beta_i    = minimize(objective, np.zeros((k+1))).x
            new_rss   = rss(ytrain, model(beta_i, train))

            #Chooses the best out of all the models
            if new_rss < old_rss:
                old_rss  = new_rss
                new_beta = beta_i
                idx = i
        
        predictors.append(predictors[k-1] + [idx]) 

        val   = xval[:,predictors[k]]
        CV[k] = kfold_validate(val, yval, lambda x: new_beta[0] + x @ new_beta[1:])
        AR2[k] = adjustedR2(val, yval, lambda x:  new_beta[0] + x @ new_beta[1:], len(new_beta)-1)
        
        betas.append(new_beta)
        index.remove(idx)

    best_idx_CV  = np.argmin(CV)
    best_idx_AR2 = np.argmax(AR2)

    print(f"Metrics: best CV model [{best_idx_CV:2}] CV = {CV[best_idx_CV]:.2f}  AR2 = {AR2[best_idx_CV]:.2f}  |  best AR2 model [{best_idx_AR2:2}] CV = {CV[best_idx_AR2]:.2f}  AR2 = {AR2[best_idx_AR2]:.2f}   --   CV: ({np.mean(CV):.2f} ± {np.std(CV,ddof=1):.2f})  AR2: ({np.mean(AR2):.2f} ± {np.std(AR2,ddof=1):.2f})")
    return betas[best_idx_CV], predictors[best_idx_CV], betas[best_idx_AR2], predictors[best_idx_AR2]

### Model selection

Forward selection is repeated several times to generate different candidate models. For each run, the models are evaluated using cross-validation (CV) MSE and adjusted \(R^2\). The CV MSE measures how well the model generalizes to unseen data, while adjusted \(R^2\) accounts for model complexity by penalizing models with unnecessary predictors.

This is important because the lowest CV MSE can sometimes be achieved by a model with a large number of parameters. Therefore, adjusted \(R^2\) is also considered when selecting the final model. By comparing both metrics, we can balance predictive performance against model complexity and select a model that provides good performance without including unnecessary predictors.


In [548]:
for i in range(10): 
    print("i = ", i)
    betaCV, predictorCV, betaR2, predictorR2 = forward_stepwise(X, y)

    output_cv = predict_save(betaCV, predictorCV, f"CV{i}", save=False)
    output_r2 = predict_save(betaR2, predictorR2, f"AR{i}", save=False)

i =  0
Metrics: best CV model [10] CV = 0.15  AR2 = 0.58  |  best AR2 model [ 5] CV = 0.15  AR2 = 0.64   --   CV: (0.18 ± 0.07)  AR2: (0.30 ± 0.31)
i =  1
Metrics: best CV model [15] CV = 0.08  AR2 = 0.79  |  best AR2 model [ 8] CV = 0.09  AR2 = 0.83   --   CV: (0.11 ± 0.11)  AR2: (0.70 ± 0.18)
i =  2
Metrics: best CV model [28] CV = 0.15  AR2 = -0.25  |  best AR2 model [ 5] CV = 0.17  AR2 = 0.68   --   CV: (0.18 ± 0.08)  AR2: (0.43 ± 0.26)
i =  3
Metrics: best CV model [12] CV = 0.07  AR2 = 0.83  |  best AR2 model [ 9] CV = 0.07  AR2 = 0.84   --   CV: (0.10 ± 0.10)  AR2: (0.70 ± 0.18)
i =  4
Metrics: best CV model [11] CV = 0.06  AR2 = 0.81  |  best AR2 model [11] CV = 0.06  AR2 = 0.81   --   CV: (0.10 ± 0.09)  AR2: (0.66 ± 0.18)
i =  5
Metrics: best CV model [12] CV = 0.07  AR2 = 0.86  |  best AR2 model [ 8] CV = 0.07  AR2 = 0.87   --   CV: (0.11 ± 0.13)  AR2: (0.73 ± 0.19)
i =  6
Metrics: best CV model [12] CV = 0.07  AR2 = 0.85  |  best AR2 model [11] CV = 0.07  AR2 = 0.85   --   C